# Next-day electricity in Sceaux, France
One historical household, one prediction per eligible date. Can seven daily meter totals tell us more than repeating yesterday? These measurements are not rent transactions, Danish readings or evidence about Kenyan tenants.

Source: G. Hébrail & A. Bérard, *Individual Household Electric Power Consumption*, UCI dataset 235, [DOI 10.24432/C58K54](https://doi.org/10.24432/C58K54), CC BY 4.0. We convert minute-average kW to daily kWh. Run all cells from this directory; set `BOMA_UCI_ZIP` if the archive is not at the default local path.

**ICS 3202 · Final engine: Ridge regression.** The user chooses an issue date; the app retrieves the preceding week ending on that date and estimates the following day's energy. The earlier `LogisticRegression_Base.ipynb` classified invoice lateness. This final module answers a different question with a different dataset: “how much energy?”, not “late or on time?”. The invoice model is not an accuracy baseline for this task.

In [1]:
import csv, hashlib, io, json, os, tempfile, zipfile
from datetime import date, timedelta
from pathlib import Path
from functools import lru_cache
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

archive = Path(os.environ.get("BOMA_UCI_ZIP", Path(tempfile.gettempdir()) / "BomaTrack_UCI_electricity.zip"))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest == "9f84b46ade8a2d8e1286ec4b2b6c2987a45a755c59f263be3b3b3d10dfbda3ff", "Check the UCI archive version"
day = timedelta(days=1)
out = Path("exports")
out.mkdir(exist_ok=True)

### What counts as a complete day?
Parse day/month/year and the meter’s recorded clock without shifting time zones. Keep a date only when every one of the 1,440 minute slots occurs exactly once with finite, non-negative active power. A blank or `?` does not become zero. Each valid minute contributes its average kW divided by 60.

In [2]:
@lru_cache(maxsize=2048)
def parse_date(s):
    d,m,y = map(int,s.split("/"))
    return date(y,m,d)

totals, seen, counts, duplicates = {}, {}, {}, 0
bad, rows = 0, 0
with zipfile.ZipFile(archive) as z, z.open("household_power_consumption.txt") as f:
    reader = csv.reader(io.TextIOWrapper(f,encoding="ascii"),delimiter=";")
    assert next(reader)[:3] == ["Date","Time","Global_active_power"]
    for ds, clock, raw, *_ in reader:
        d = parse_date(ds)
        if d not in seen:
            seen[d], counts[d], totals[d] = bytearray(1440), 0, 0.0
        minute = int(clock[:2])*60 + int(clock[3:5])
        assert clock.endswith(":00") and 0 <= minute < 1440
        rows += 1
        if seen[d][minute]:
            duplicates += 1
            seen[d][minute] = 2
        else:
            seen[d][minute] = 1
        if raw in ("","?"):
            bad += 1
            continue
        kw = float(raw)
        assert np.isfinite(kw) and kw >= 0
        totals[d] += kw / 60
        counts[d] += 1
daily = {d:kwh for d,kwh in totals.items() if counts[d]==1440 and all(n==1 for n in seen[d])}
print(f"1. {rows:,} minute records across {len(seen):,} recorded dates ({min(seen)} to {max(seen)}).")
print(f"2. {bad:,} active-power readings missing; {duplicates} duplicate minute keys.")
print(f"3. {len(daily):,} complete days; {len(seen)-len(daily)} incomplete days excluded.")
print(f"4. Complete daily use ranges from {min(daily.values()):.2f} to {max(daily.values()):.2f} kWh (peaks retained).")

1. 2,075,259 minute records across 1,442 recorded dates (2006-12-16 to 2010-11-26).
2. 25,979 active-power readings missing; 0 duplicate minute keys.
3. 1,358 complete days; 84 incomplete days excluded.
4. Complete daily use ranges from 4.17 to 79.56 kWh (peaks retained).


### Build a seven-day window, then split by the target date
The forecast issued at the end of day *t* uses days *t−6* through *t*, oldest first. Its label is *t+1*. Looking up dates explicitly prevents a missing day from being skipped. Readings from earlier test dates may enter history after they have been observed.

In [3]:
dates = sorted(d for d in daily if all(d-i*day in daily for i in range(7,0,-1)))
X = np.array([[daily[d-i*day] for i in range(7,0,-1)] for d in dates])
y = np.array([daily[d] for d in dates])
year = np.array([d.year for d in dates])
train, val, test = year <= 2008, year == 2009, year == 2010
assert (train.sum(),val.sum(),test.sum()) == (467,232,210)
print(f"5. Requiring a complete previous week leaves {len(dates)} eligible targets; {len(daily)-len(dates)} otherwise complete targets drop out.")
for name,mask in [("Train",train),("Validation",val),("Test",test)]:
    selected=[d for d,ok in zip(dates,mask) if ok]
    print(name,len(selected),selected[0],selected[-1])

5. Requiring a complete previous week leaves 909 eligible targets; 449 otherwise complete targets drop out.
Train 467 2006-12-29 2008-12-31
Validation 232 2009-01-01 2009-12-31
Test 210 2010-01-01 2010-11-25


### Select on 2009, not on the final test
Ridge makes a weighted sum of the seven daily readings. First we put each day's input on a common scale: subtract its training average and divide by its training spread. Ridge's penalty discourages very large weights so a few unusual weeks do not dominate the fit. Validation is a rehearsal on later dates: whichever setting has the smallest average miss wins. A shallow tree is a different approach that splits past weeks into groups using a few thresholds.

Compare Ridge penalties 0.1, 1 and 10 (standardised inputs) and a depth-three tree. Fit scaling on training rows only. Select by validation MAE, with Ridge first in a tie. Negative forecasts are floored at zero, consistently for every method.

In [4]:
def candidates():
    return {**{f"ridge-{a}":make_pipeline(StandardScaler(),Ridge(alpha=a)) for a in (0.1,1,10)},
            "tree-depth3":DecisionTreeRegressor(max_depth=3,random_state=3102)}
def scores(obs, pred):
    pred=np.maximum(0,pred)
    return {"MAE_kWh":float(mean_absolute_error(obs,pred)),"RMSE_kWh":float(root_mean_squared_error(obs,pred))}
valid = {name:scores(y[val],model.fit(X[train],y[train]).predict(X[val])) for name,model in candidates().items()}
chosen = min(valid,key=lambda name:valid[name]["MAE_kWh"])
print("Validation:", json.dumps(valid,indent=2), "\nSelected:",chosen)

Validation: {
  "ridge-0.1": {
    "MAE_kWh": 4.378797801950257,
    "RMSE_kWh": 5.6417336232011595
  },
  "ridge-1": {
    "MAE_kWh": 4.377866607756424,
    "RMSE_kWh": 5.641391447587671
  },
  "ridge-10": {
    "MAE_kWh": 4.36940988251584,
    "RMSE_kWh": 5.6392464294736175
  },
  "tree-depth3": {
    "MAE_kWh": 4.995386498613932,
    "RMSE_kWh": 6.409844850345919
  }
} 
Selected: ridge-10


### Freeze the selected model and measure the final error
Refit on the 699 development targets (training + validation). No test row is used to fit coefficients, scaling or tree splits. Evaluate the chosen model and comparisons on the same 210 later dates. MAE is the average size of a miss in kWh; RMSE gives large misses extra weight.

In [5]:
best_ridge = min((n for n in valid if n.startswith("ridge")),key=lambda n:valid[n]["MAE_kWh"])
models = {n:m.fit(X[train|val],y[train|val]) for n,m in candidates().items() if n in (best_ridge,"tree-depth3")}
predictions = {"Yesterday":X[test,-1],"Same weekday last week":X[test,0],
               **{name:model.predict(X[test]) for name,model in models.items()}}
final = {n:scores(y[test],pred) for n,pred in predictions.items()}
winner = min(final,key=lambda name:final[name]["MAE_kWh"])
print(json.dumps(final,indent=2))
print(f"Lowest held-out MAE: {winner} ({final[winner]['MAE_kWh']:.2f} kWh).")
print("These scores describe this French home only; they do not measure accuracy for a new tenant or building.")
with open("heldout_predictions.csv","w",newline="",encoding="utf8") as f:
    w=csv.writer(f); w.writerow(["target_date","actual_kWh",*predictions])
    w.writerows(zip(np.array(dates)[test],y[test],*(np.maximum(0,p) for p in predictions.values())))

{
  "Yesterday": {
    "MAE_kWh": 5.1200593650793635,
    "RMSE_kWh": 7.163946301284865
  },
  "Same weekday last week": {
    "MAE_kWh": 5.845522222222222,
    "RMSE_kWh": 8.003040559155089
  },
  "ridge-10": {
    "MAE_kWh": 4.463954855339591,
    "RMSE_kWh": 6.072552304128218
  },
  "tree-depth3": {
    "MAE_kWh": 4.809044321448548,
    "RMSE_kWh": 6.24038621404636
  }
}
Lowest held-out MAE: ridge-10 (4.46 kWh).
These scores describe this French home only; they do not measure accuracy for a new tenant or building.


### Export parameters and measurements, not a lookup table of forecasts
Android needs the learned coefficients, input order, standardisation and output floor. The mobile CSV contains dated measurements (with blank incomplete days); the expected Python predictions are kept **outside** the app. To avoid choosing flattering examples, use the first, middle and last eligible test targets as demo shortcuts and the first held-out issue date with a gap.

In [6]:
assert chosen == best_ridge, "If the tree wins validation, review the model and mobile implementation before exporting."
model=models[chosen]
scaler,ridge=model.steps[0][1],model.steps[1][1]
version="sceaux-ridge-v1"
fields={"version":version,"model":"ridge","units":"kWh","source_doi":"10.24432/C58K54","dataset_sha256":digest,
        "features":",".join(f"target_minus_{i}" for i in range(7,0,-1)),"output_floor":"0",
        "alpha":str(ridge.alpha),"intercept":repr(float(ridge.intercept_))}
for k,array in [("mean",scaler.mean_),("scale",scaler.scale_),("weights",ridge.coef_)]:
    fields[k]=",".join(repr(float(x)) for x in array)
def days_between(start,end):
    return (start+i*day for i in range((end-start).days+1))
test_dates=[d for d,ok in zip(dates,test) if ok]
targets=[test_dates[0],test_dates[len(test_dates)//2],test_dates[-1]]
gap=next(t-day for t in days_between(test_dates[0],test_dates[-1]) if t in daily and t not in test_dates)
fields["issue_shortcuts"]=",".join(str(t-day) for t in targets)+","+str(gap)
(out/"french_ridge.properties").write_text("".join(f"{k}={v}\n" for k,v in fields.items()),encoding="ascii")
with (out/"french_daily.csv").open("w",newline="",encoding="ascii") as f:
    w=csv.writer(f,lineterminator="\n"); w.writerow(["date","measured_kWh"])
    for d in days_between(test_dates[0]-7*day,test_dates[-1]):
        w.writerow([d,repr(daily[d]) if d in daily else ""])
checks=[]
for t in targets:
    n=dates.index(t); pred=float(max(0,model.predict(X[n:n+1])[0]))
    checks.append(dict(issue=str(t-day),target=str(t),history=X[n].tolist(),prediction=pred,actual=float(y[n]),
                       difference=float(y[n])-pred))
report=dict(version=version,archive_sha256=digest,selected=chosen,validation=valid,test=final,
            winner_on_test=winner,examples=checks,gap_issue=str(gap),split=[int(m.sum()) for m in (train,val,test)])
Path("evaluation.json").write_text(json.dumps(report,indent=2)+"\n",encoding="utf8")
for c in checks: print(f"Issue {c['issue']} -> {c['target']}: {c['prediction']:.2f} kWh predicted, {c['actual']:.2f} kWh recorded.")
print("Gap example (no prediction):",gap)

Issue 2009-12-31 -> 2010-01-01: 34.77 kWh predicted, 20.40 kWh recorded.
Issue 2010-06-25 -> 2010-06-26: 22.86 kWh predicted, 16.36 kWh recorded.
Issue 2010-11-24 -> 2010-11-25: 26.90 kWh predicted, 23.85 kWh recorded.
Gap example (no prediction): 2010-01-02


### Why do the two app results differ?
The phone starts at a learned reference forecast, then adds one adjustment for each of the seven dated readings. A reading contributes its standardised difference from the training average multiplied by its learned weight. Below, we reconstruct the phone's two examples directly from the exported parameters. These are contributions to the model's arithmetic, not proof that a particular day or appliance caused future use.

In [7]:
base = float(fields["intercept"])
means, spreads, weights = (np.fromstring(fields[key], sep=",") for key in ("mean", "scale", "weights"))
for case in checks[:2]:
    changes = (np.asarray(case["history"]) - means) / spreads * weights
    rebuilt = max(0.0, base + float(changes.sum()))
    assert np.isclose(rebuilt, case["prediction"])
    print(f"{case['issue']}: reference {base:.2f} kWh; all seven adjustments {changes.sum():+.2f} kWh; forecast {rebuilt:.2f} kWh.")
    print(f"  Latest reading {case['history'][-1]:.2f} kWh contributed {changes[-1]:+.2f} kWh of that adjustment.")
gap_day = date.fromisoformat(report["gap_issue"])
missing = [d for d in days_between(gap_day - 6*day, gap_day) if d not in daily]
assert missing
print("Insufficient history for", gap_day + day, "because a complete daily reading is missing on", *missing)

2009-12-31: reference 26.01 kWh; all seven adjustments +8.76 kWh; forecast 34.77 kWh.
  Latest reading 36.86 kWh contributed +3.38 kWh of that adjustment.
2010-06-25: reference 26.01 kWh; all seven adjustments -3.14 kWh; forecast 22.86 kWh.
  Latest reading 18.32 kWh contributed -2.32 kWh of that adjustment.
Insufficient history for 2010-01-03 because a complete daily reading is missing on 2010-01-02


### Interpretation and limits
The Ridge model missed by 4.46 kWh on average across 210 held-out dates; simply repeating yesterday missed by 5.12 kWh. An individual error can be much larger: the first example overshot by 14.36 kWh. The next-day meter value is revealed only for comparison, never fed to the model. A week with a missing day yields no forecast. This study covers one French household; it does not establish performance for other homes.